# [재실행 전용 · 2026-10-05 회계 수정] 6.8절 top-K 스윕 재평가

학습 셀 없이, 이미 저장된 K=5/7/15 모델(`models/crypto_hrl_pool_v4_topk_sweep/agent_beta_-30_k{K}.zip`)과
K=10 배포 모델(`models/crypto_hrl_pool_v4/agent_beta_-30.zip`)을 회계 수정 후 환경(보유 비중 drift, 거래 불가 종목 마스크)으로 다시 평가한다.

- §17-C: K별 valid(2025 Q4)/test(2026~) 성과와 샤프 격차 (논문 6.8절 표)
- §17-A: top-K 확신도 로짓과 실현 수익률의 스피어만 IC (β별 + K별)

위에서부터 **순서대로 전부 실행**하면 된다(학습 없음, CPU). 결과 CSV는 `data/crypto/`에 `_fix1004` 접미사로 저장된다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q stable-baselines3[extra] gymnasium h5py pyarrow


In [ ]:
import warnings
import glob
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from dataclasses import dataclass

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO, DQN
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.monitor import Monitor

warnings.filterwarnings('ignore')
np.random.seed(42)
plt.rcParams['axes.unicode_minus'] = False

DRIVE_ROOT = Path('/content/drive/MyDrive/졸업프로젝트')
DATA_DIR   = DRIVE_ROOT / 'data'
CRYPTO_DIR = DATA_DIR / 'crypto'
FEAT_DIR   = CRYPTO_DIR / 'features_5m'
MODEL_DIR  = DRIVE_ROOT / 'models'
POOL_DIR   = MODEL_DIR / 'crypto_hrl_pool'

RL_EMB_H5      = CRYPTO_DIR / 'crypto_rl_embeddings.h5'
ORIG_PPO_PATH  = MODEL_DIR / 'crypto_ppo_portfolio.zip'   # crypto_rl_trading_colab.ipynb 산출물(있으면 비교에 포함)

EMB_DIM    = 64
TC         = 0.001    # 거래비용 (Binance taker fee 기준)
BUCKET_SEC = 1800     # 30분 버킷

RL_TRAIN_START = '2025-01-01'
RL_TRAIN_END   = '2025-12-31'
RL_TEST_START  = '2026-01-01'
RL_TEST_END    = '2099-12-31'   # 열린 구간

# --- Stage A/B/C 하이퍼파라미터 ---
N_LABELS              = 5          # 레짐 라벨 수 (bear/pullback/sideways/rally/bull)
RISK_THRESHOLD        = 0.2        # EarnHFT θ — 상하위 라벨 경계 분위수 폭
LOWPASS_WINDOW        = 48         # 레짐 분할 저역통과 윈도우 = 1일(30분×48)
POOL_BETAS            = [-90, -30, 30, 90]   # Stage B 선호도 스윕 (논문 [-90,-10,30,100] 근사)
POOL_TIMESTEPS_EACH   = 100_000    # 풀 에이전트 1개당 학습 스텝 (원본 300k와 총량 비슷하게)
ROUTE_BUCKETS         = 48         # Stage C 라우팅 단위 = 1일
MACRO_WINDOW           = 96        # 라우터 상태에 포함할 최근 버킷 수 (=2일)
ROUTER_TOTAL_TIMESTEPS = 20_000    # 라우터(=일 단위) 학습 스텝

assert RL_EMB_H5.exists(), 'crypto_rl_embeddings.h5 없음 — crypto_hybrid_model_colab.ipynb 먼저 실행'
print('경로 확인 완료')


In [ ]:
print('crypto_rl_embeddings.h5 로드 중...')
with h5py.File(RL_EMB_H5, 'r') as f:
    symbols_raw = f['symbols'][:]
    ts_raw      = f['timestamps'][:]     # epoch seconds (UTC)
    embs_all    = f['embeddings'][:]     # (N, 64)

symbols_list = [s.decode() for s in symbols_raw]
ts_list      = ts_raw.astype(np.int64).tolist()

# (symbol, 30분 버킷) → 임베딩 인덱스
emb_index  = {}
sym_ts_set = {}   # 심볼별로 임베딩 추출에 실제 쓰인 초 단위 timestamp 집합 (수익률 정확 매칭용)
for i, (s, t) in enumerate(zip(symbols_list, ts_list)):
    emb_index[(s, t // BUCKET_SEC)] = i
    sym_ts_set.setdefault(s, set()).add(t)

symbols  = sorted(set(symbols_list))
N_ASSETS = len(symbols)
print(f'임베딩: {embs_all.shape} | 심볼: {N_ASSETS}개 | 인덱스: {len(emb_index):,}개')


In [ ]:
print('features_5m parquet에서 RetTarget_6b 로드 중...')
files = sorted(glob.glob(str(FEAT_DIR / '*.parquet')))
assert files, f'피처 parquet 없음: {FEAT_DIR}'

rl_start_ts = pd.Timestamp(RL_TRAIN_START, tz='UTC')
ret_index   = {}

for fp in files:
    sym = Path(fp).stem
    valid_ts = sym_ts_set.get(sym)
    if not valid_ts:
        continue
    df = pd.read_parquet(fp, columns=['datetime', 'RetTarget_6b'])
    df = df[df['datetime'] >= rl_start_ts]
    epoch = df['datetime'].values.astype('int64') // 10**9   # ts_list와 동일 단위(초)
    y     = df['RetTarget_6b'].values
    # 임베딩 추출에 실제 쓰인 timestamp만 매칭 — 같은 버킷 내 다른 5분봉과 섞이지 않도록
    for t, r in zip(epoch, y):
        t = int(t)
        if t in valid_ts and not np.isnan(r):
            ret_index[(sym, t // BUCKET_SEC)] = float(r)

all_buckets = sorted(set(t // BUCKET_SEC for t in ts_list))
print(f'수익률 인덱스: {len(ret_index):,}개 | 전체 30분 버킷: {len(all_buckets):,}개')


In [ ]:
# [2026-10-04 회계 수정] 가격 변동 후 실제 보유 비중(drift)·거래 가능 종목 마스크.
# 이전 버전은 수익 반영 뒤에도 직전 '목표' 비중을 보유 비중으로 저장해, 가격 변동으로 틀어진 비중을
# 되돌리는 재조정 거래(등가중 포함)의 비용이 0으로 계산됐다.
def _drift_weights(w, rets):
    """수익 반영 뒤 실제 보유 비중 w_i(1+r_i)/(1+wᵀr). 합<1이면 나머지는 현금(수익률 0)으로 남는다."""
    w = np.asarray(w, dtype=np.float64)
    g = 1.0 + float(np.dot(w, rets))
    if g <= 1e-12:
        return w.astype(np.float32)
    return (w * (1.0 + np.asarray(rets, dtype=np.float64)) / g).astype(np.float32)


def _mask_untradable(w, tradable, renorm=True):
    """수익률 데이터가 없는(상장 전·상폐 후) 종목 비중을 0으로. renorm=False면 빠진 몫은 현금으로 남는다."""
    w = np.where(tradable, np.asarray(w, dtype=np.float64), 0.0)
    s = w.sum()
    return (w / s if renorm and s > 1e-12 else w).astype(np.float32)


class CryptoPortfolioEnv(gym.Env):
    """
    크립토 44종목 개별 자산 기반 포트폴리오 운용 환경 (클러스터링 없음)

    State:  종목별 임베딩(N×64) + 현재 비중(N) → N*64+N dim
    Action: 종목별 목표 포트폴리오 비중 (N-dim, Softmax 정규화)
    Reward: 30분 포트폴리오 수익률(RetTarget_6b, 이미 forward-looking) − 거래비용
    스텝 = 30분 버킷(BUCKET_SEC) 단위 — emb_index/ret_index 모두 (symbol, bucket) 키.
    """
    metadata = {'render_modes': []}

    def __init__(self, buckets, symbols, emb_index, embs_all, ret_index,
                 emb_dim=64, tc=0.001):
        super().__init__()
        self.buckets    = buckets
        self.symbols    = symbols
        self.n_assets   = len(symbols)
        self.emb_index  = emb_index
        self.embs_all   = embs_all
        self.ret_index  = ret_index
        self.emb_dim    = emb_dim
        self.tc         = tc

        obs_dim = self.n_assets * emb_dim + self.n_assets
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(obs_dim,), dtype=np.float32)
        self.action_space      = spaces.Box(-3.0, 3.0, shape=(self.n_assets,), dtype=np.float32)

    def _asset_emb(self, bucket):
        """종목별 임베딩 — 해당 버킷에 데이터 없으면 0-vector(상장 전/상폐 등)"""
        embs = []
        for sym in self.symbols:
            idx = self.emb_index.get((sym, bucket))
            embs.append(self.embs_all[idx] if idx is not None else np.zeros(self.emb_dim, dtype=np.float32))
        return np.concatenate(embs)  # (N*64,)

    def _asset_ret(self, bucket):
        """종목별 30분 수익률 — 없으면 0(그 자산은 보유해도 손익 없음으로 취급)"""
        return np.array([self.ret_index.get((sym, bucket), 0.0) for sym in self.symbols], dtype=np.float32)

    def _tradable(self, bucket):
        """해당 버킷에 수익률 데이터가 있는(거래 가능한) 종목 마스크"""
        return np.array([(sym, bucket) in self.ret_index for sym in self.symbols])

    def _get_obs(self):
        b = self.buckets[self.t]
        return np.concatenate([self._asset_emb(b), self.weights]).astype(np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.t               = 0
        self.weights         = _mask_untradable(np.ones(self.n_assets, dtype=np.float32),
                                          self._tradable(self.buckets[0]))  # 거래 가능 종목 등가중에서 시작
        self.portfolio_value = 1.0
        self.value_history   = [1.0]
        self.turnover_history = []
        return self._get_obs(), {}

    def step(self, action):
        # Softmax로 비중 정규화
        action  = np.exp(action - action.max())
        action  = action / action.sum()
        action  = _mask_untradable(action, self._tradable(self.buckets[self.t]))  # 거래 불가 종목 제외

        # 거래비용
        turnover = float(np.abs(action - self.weights).sum())  # 비중 변화량(진단용)
        tc_cost  = self.tc * turnover

        # RetTarget_6b는 현재 버킷 자체에 forward 수익률이 이미 인코딩되어 있음
        b          = self.buckets[self.t]
        asset_rets = self._asset_ret(b)
        port_ret   = float((action * asset_rets).sum())

        self.portfolio_value *= (1 + port_ret - tc_cost)
        self.weights           = _drift_weights(action, asset_rets)  # 가격 변동 후 실제 보유 비중
        self.value_history.append(self.portfolio_value)
        self.turnover_history.append(turnover)
        self.t                += 1

        done = (self.t >= len(self.buckets) - 1)
        reward = float(port_ret - tc_cost)

        info = {'portfolio_value': self.portfolio_value, 'port_ret': port_ret, 'turnover': turnover, 'tc_cost': tc_cost}
        return self._get_obs(), reward, done, False, info


In [ ]:
def bucket_to_ts(b):
    return pd.Timestamp(b * BUCKET_SEC, unit='s', tz='UTC')

def bucket_to_date(b):
    return bucket_to_ts(b).strftime('%Y-%m-%d')

train_buckets = [b for b in all_buckets if RL_TRAIN_START <= bucket_to_date(b) <= RL_TRAIN_END]
test_buckets  = [b for b in all_buckets if RL_TEST_START  <= bucket_to_date(b) <= RL_TEST_END]
print(f'학습 기간: {bucket_to_ts(train_buckets[0])} ~ {bucket_to_ts(train_buckets[-1])} ({len(train_buckets):,}스텝)')
print(f'테스트 기간: {bucket_to_ts(test_buckets[0])} ~ {bucket_to_ts(test_buckets[-1])} ({len(test_buckets):,}스텝)')

env_kwargs = dict(
    symbols=symbols, emb_index=emb_index, embs_all=embs_all,
    ret_index=ret_index, emb_dim=EMB_DIM, tc=TC,
)

test_env = CryptoPortfolioEnv(test_buckets, **env_kwargs)

print('\n환경 유효성 검사...')
check_env(CryptoPortfolioEnv(train_buckets[:100], **env_kwargs), warn=True)
print('OK')


In [ ]:
TOP_K = 10   # 종목 유니버스(44) 중 매 스텝 보유 가능한 최대 종목 수 (FreQuant top-|G| 이식)

class SparseGatedCryptoPortfolioEnv(CryptoPortfolioEnv):
    """
    FreQuant(top-K 희소 선택) + DeepClair(전용 리스크게이트 ρ) 결합 저수준 환경.

    Action: (N+1,)-dim — 앞 N개는 종목별 confidence logit(FreQuant Eq.13-14의 h_i 역할),
            마지막 1개는 리스크자산 총 노출 게이트 로짓(DeepClair의 ρ_t 역할).
    변환:   top-K confidence logit만 골라 그 안에서 softmax(FreQuant Eq.14) →
            sigmoid(게이트)를 곱해 최종 종목별 비중. 나머지는 암묵적 현금(수익률 0).
    관측·NAV 회계는 CryptoPortfolioEnv와 동일. step_with_weights()는 목표 비중을 직접
    받아 NAV를 갱신하는 보조 메서드 — §13-G 관성 스무딩에서 (top-K 변환을 다시 태우지 않고)
    이미 블렌딩된 비중을 그대로 반영하기 위해 필요하다.
    """

    def __init__(self, *args, top_k=TOP_K, **kwargs):
        self.top_k = top_k
        super().__init__(*args, **kwargs)
        self.action_space = spaces.Box(-3.0, 3.0, shape=(self.n_assets + 1,), dtype=np.float32)

    def action_to_weights(self, action):
        action = np.asarray(action, dtype=np.float64)
        asset_logits, gate_logit = action[:-1], action[-1]
        asset_logits = np.where(self._tradable(self.buckets[self.t]), asset_logits, -np.inf)  # 거래 불가 종목 제외
        k = min(self.top_k, self.n_assets)
        top_idx = np.argpartition(asset_logits, -k)[-k:]
        sel = asset_logits[top_idx]
        sel = np.exp(sel - sel.max())
        sel = sel / sel.sum()
        risky_w = np.zeros(self.n_assets, dtype=np.float32)
        risky_w[top_idx] = sel
        rho = float(1.0 / (1.0 + np.exp(-gate_logit)))
        return (rho * risky_w).astype(np.float32), rho, top_idx

    def reset(self, seed=None, options=None):
        obs, info = super().reset(seed=seed, options=options)
        self.rho_history = []
        self.selection_history = []
        return obs, info

    def step_with_weights(self, target_w, extra_info=None):
        target_w = _mask_untradable(target_w, self._tradable(self.buckets[self.t]), renorm=False)  # 빠진 몫은 현금
        turnover = float(np.abs(target_w - self.weights).sum())
        tc_cost  = self.tc * turnover
        b          = self.buckets[self.t]
        asset_rets = self._asset_ret(b)
        port_ret   = float((target_w * asset_rets).sum())

        self.portfolio_value *= (1 + port_ret - tc_cost)
        self.weights           = _drift_weights(target_w, asset_rets)  # 가격 변동 후 실제 보유 비중
        self.value_history.append(self.portfolio_value)
        self.turnover_history.append(turnover)
        self.t                += 1

        done = (self.t >= len(self.buckets) - 1)
        reward = float(port_ret - tc_cost)
        info = {'portfolio_value': self.portfolio_value, 'port_ret': port_ret,
                'turnover': turnover, 'tc_cost': tc_cost}
        if extra_info:
            info.update(extra_info)
            if 'rho' in extra_info:
                self.rho_history.append(extra_info['rho'])
            if 'selected' in extra_info:
                self.selection_history.append(extra_info['selected'])
        return self._get_obs(), reward, done, False, info

    def step(self, action):
        target_w, rho, top_idx = self.action_to_weights(action)
        return self.step_with_weights(target_w, extra_info={'rho': rho, 'selected': top_idx.tolist()})


# 스모크 테스트
_sg_env = SparseGatedCryptoPortfolioEnv(test_buckets[:200], top_k=TOP_K, **env_kwargs)
check_env(_sg_env, warn=True)
obs, _ = _sg_env.reset()
a = np.random.randn(_sg_env.n_assets + 1).astype(np.float32)
obs, r, done, _, info = _sg_env.step(a)
print(f"action space: {_sg_env.action_space.shape} | 선택된 종목 수: {len(info['selected'])} | "
      f"rho: {info['rho']:.3f} | reward: {r:.5f}")

In [ ]:
# 모델 로드 — K=10은 배포 모델(§4.6), 나머지는 §17-C에서 학습해 둔 스윕 모델
TOPK_SWEEP = [5, 7, 10, 15]
POOL_DIR_V4      = MODEL_DIR / 'crypto_hrl_pool_v4'
POOL_DIR_V4_TOPK = MODEL_DIR / 'crypto_hrl_pool_v4_topk_sweep'

pool_models_v4 = {b: PPO.load(str(POOL_DIR_V4 / f'agent_beta_{b}.zip')) for b in POOL_BETAS}
topk_models = {10: pool_models_v4[-30]}
for k in TOPK_SWEEP:
    if k == 10:
        continue
    p = POOL_DIR_V4_TOPK / f'agent_beta_-30_k{k}.zip'
    assert p.exists(), f'모델 없음: {p}'
    topk_models[k] = PPO.load(str(p))
print('로드 완료 — β 풀:', list(pool_models_v4.keys()), '| K:', sorted(topk_models.keys()))

valid_buckets = [b for b in train_buckets if '2025-10-01' <= bucket_to_date(b) <= '2025-12-31']
print(f'valid {len(valid_buckets):,}스텝 | test {len(test_buckets):,}스텝')


## §17-C — K별 valid/test 성과와 샤프 격차 (논문 6.8절 표)

수정 전 값(논문 현재 표): K=5 1.741/0.541, K=7 −0.516/1.593, K=10 1.314/1.156, K=15 −0.200/0.734 (valid/test 샤프). 수정 후 K=10은 4.6절 β=−30 값(0.914/1.310)과 같아야 한다.


In [ ]:
ALPHA_DEPLOY = 0.01
STEPS_PER_YEAR = 48 * 365


def run_solo_smoothed_topk(model, alpha, buckets, top_k):
    env = SparseGatedCryptoPortfolioEnv(buckets=buckets, top_k=top_k, **env_kwargs)
    obs, _ = env.reset()
    done = False
    while not done:
        a, _ = model.predict(obs, deterministic=True)
        target_w, rho, top_idx = env.action_to_weights(a)
        w = (1 - alpha) * env.weights + alpha * target_w
        obs, _, done, _, _ = env.step_with_weights(w, extra_info={'rho': rho, 'selected': top_idx.tolist()})
    return np.array(env.value_history), np.array(env.turnover_history)


def perf(net):
    rets = np.diff(net) / net[:-1]
    peak = np.maximum.accumulate(net)
    return {
        'ret': net[-1] / net[0] - 1,
        'sharpe': rets.mean() / (rets.std() + 1e-8) * np.sqrt(STEPS_PER_YEAR),
        'mdd': (net / peak - 1).min(),
    }


rows = {}
for k in TOPK_SWEEP:
    v_net, v_turn = run_solo_smoothed_topk(topk_models[k], ALPHA_DEPLOY, valid_buckets, k)
    t_net, t_turn = run_solo_smoothed_topk(topk_models[k], ALPHA_DEPLOY, test_buckets, k)
    v, t = perf(v_net), perf(t_net)
    rows[k] = {
        'valid Sharpe': v['sharpe'], 'test Sharpe': t['sharpe'],
        '|valid-test| 격차': abs(v['sharpe'] - t['sharpe']),
        'valid 수익률': v['ret'], 'test 수익률': t['ret'],
        'valid MDD': v['mdd'], 'test MDD': t['mdd'],
        'test 평균 턴오버': float(np.mean(t_turn)) if len(t_turn) else np.nan,
    }
    print(f"K={k:>2d}: valid {v['sharpe']:+.3f} ({v['ret']:+.2%}) | test {t['sharpe']:+.3f} ({t['ret']:+.2%}) | 격차 {abs(v['sharpe']-t['sharpe']):.3f}")

topk_gap_df = pd.DataFrame(rows).T
topk_gap_df.index.name = 'K'
print()
print(topk_gap_df.to_string(float_format=lambda x: f'{x:.4f}'))

# K=10 = 배포 모델이므로 4.6절 재계산 값과 맞는지 확인
assert abs(rows[10]['valid Sharpe'] - 0.914) < 0.01 and abs(rows[10]['test Sharpe'] - 1.310) < 0.01, \
    'K=10이 4.6절 β=-30 수정 후 값(0.914/1.310)과 다름 — 환경 정의를 확인할 것'
topk_gap_df.to_csv(CRYPTO_DIR / 'crypto_hrl_v4_topk_sweep_gap_fix1004.csv', encoding='utf-8-sig')
print('\n저장 완료: crypto_hrl_v4_topk_sweep_gap_fix1004.csv')


## §17-A — 확신도 로짓 vs 실현 수익률 스피어만 IC

수정 전 값: β별 평균 IC 0.0006–0.0057, IC>0 비율 50.3–51.5%, β=−90만 t=3.33. 관측의 직전 비중이 drift 반영값으로 바뀌어 수치가 약간 달라질 수 있다.


In [ ]:
from scipy.stats import spearmanr


def compute_step_ic(model, buckets, top_k=TOP_K):
    """스텝별 (원시 확신도 로짓, 실현 수익률) 스피어만 상관. 수익률이 없는 종목은 결측으로 제외.
    원본 §17-A와 같이 스무딩 없이 목표 비중으로 진행한다."""
    env = SparseGatedCryptoPortfolioEnv(buckets=buckets, top_k=top_k, **env_kwargs)
    obs, _ = env.reset()
    ics, done = [], False
    while not done:
        a, _ = model.predict(obs, deterministic=True)
        logits = np.asarray(a[:-1], dtype=np.float64)
        b = env.buckets[env.t]
        realized = np.array([ret_index.get((sym, b), np.nan) for sym in symbols])
        mask = ~np.isnan(realized)
        if mask.sum() >= 5:
            r, _ = spearmanr(logits[mask], realized[mask])
            if not np.isnan(r):
                ics.append(r)
        target_w, rho_gate, top_idx = env.action_to_weights(a)
        obs, _, done, _, _ = env.step_with_weights(target_w, extra_info={'rho': rho_gate, 'selected': top_idx.tolist()})
    return np.array(ics)


def ic_row(ic):
    return {'평균 IC': ic.mean(), 'IC 표준편차': ic.std(), 'IC>0 비율': (ic > 0).mean(),
            't통계량': ic.mean() / (ic.std() / np.sqrt(len(ic)) + 1e-12), 'n_steps': len(ic)}


ic_rows = {}
for beta in POOL_BETAS:
    ic_rows[f'β={beta} (K=10)'] = ic_row(compute_step_ic(pool_models_v4[beta], test_buckets, 10))
for k in TOPK_SWEEP:
    if k != 10:
        ic_rows[f'β=-30 (K={k})'] = ic_row(compute_step_ic(topk_models[k], test_buckets, k))

ic_df = pd.DataFrame(ic_rows).T
print(ic_df.to_string(float_format=lambda x: f'{x:.4f}'))
ic_df.to_csv(CRYPTO_DIR / 'crypto_hrl_v4_topk_ic_diag_fix1004.csv', encoding='utf-8-sig')
print('\n저장 완료: crypto_hrl_v4_topk_ic_diag_fix1004.csv')
